In [ ]:
#This code takes the original data, the prediction of nns and the BMS, computes the rmse and mae for interpolation and extrapolation
#and saves everything into a dataframe 

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from matplotlib.pyplot import figure
import seaborn as sns
import matplotlib.gridspec as gridspec
import ast
import sys
sys.path.append('../no_degeneracy/')
sys.path.append('../no_degeneracy/Prior/')
from mcmc import *
from parallel import *
from fit_prior import read_prior_par
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error

In [2]:
def clean_index(dataframe):
    dataframe.set_index('Unnamed: 0', inplace=True)
    dataframe.index.name = None
    dataframe= dataframe.reset_index(drop=True)
    return dataframe

def add_bms_pred(dataframe, bms_trace, number_param):
    VARS = ['x1',]
    x = dataframe[[c for c in VARS]].copy()
    y=dataframe.y

    if number_param==10:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np10.2017-10-18 18:07:35.089658.dat')
    elif number_param==20:
        prior_par = read_prior_par('../no_degeneracy/Prior/final_prior_param_sq.named_equations.nv1.np20.maxs200.2024-05-10 162907.551306.dat')

    #mdl model
    minrow = bms_trace[bms_trace.H == min(bms_trace.H)].iloc[0]
    minH, minexpr, minparvals = minrow.H, minrow.expr, ast.literal_eval(minrow.parvals)

    t = Tree(
        variables=list(x.columns),
        parameters=['a%d' % i for i in range(number_param)],
        x=x, y=y,
        prior_par=prior_par,
        max_size=200,
        from_string=minexpr,
    )

    t.set_par_values(deepcopy(minparvals))

    dplot = deepcopy(dataframe)
    dplot['ybms'] = t.predict(x)
    dinterpolate=deepcopy(dplot)


    return dinterpolate
    

In [3]:
#Read NN and BMS data
#resolutions={'0.5x':'0.1', '1x':'0.05', '2x': '0.025', '4e-3x':'0.004' }
#resolution='1x'

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
resolution='4e-3x' #'0.5x', '1x', '2x', '4e-3'
resolutions_interp={'0.5x':'0.02', '1x':'0.01' , '2x': '0.005' , '4e-3x':'0.0008' }

resolutions={'0.5x':'0.1', '1x':'0.05' , '2x': '0.025' , '4e-3x':'0.004' }
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


functions=['tanh', 'leaky_ReLU'] #tanh, leaky_ReLU
realizations=2
N=9
sigmas=[0.0, 0.02, 0.04,0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.20]


runid=0
NPAR=10 #10, 20
steps=50000


rmse_nn_train=[];rmse_nn_test=[]
rmse_mdl_train=[];rmse_mdl_test=[]

mae_nn_train=[];mae_nn_test=[]
mae_mdl_train=[];mae_mdl_test=[]

n_index=[];r_index=[];sigma_index=[];function_index=[]

#Put mae and rmse of each simulation (on nn and bms) in a dataframe
for function in functions:
    for sigma in sigmas:
        for realization in range(realizations+1):
            
            #Read NN data
            #file_model='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(r) + '.csv'
            #file_model_hr='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(r) + '_res_0.01.csv'

            #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            file_model='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(realization) + '.csv'
            model_d='../../data/nns/' + resolution + '_resolution/approximation/' + file_model
            
            
            file_model_hr='NN_no_overfit_' + function + '_sigma_' + str(sigma) + '_r_' + str(realization) + '_res_' + \
            resolutions_interp[resolution] + '.csv'
            model_dhr='../../data/nns/' + resolution + '_resolution/inter_extrapolation/' + file_model_hr
            #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
            
            #model_dhr='../data/inter_extrapolate_nns/' + file_model_hr
            dhr=pd.read_csv(model_dhr)

            #model_d='../data/1x_resolution/trained_nns/' + file_model
            d=pd.read_csv(model_d)
            

            for n in range(N+1):
                n_index.append(n);r_index.append(realization);sigma_index.append(sigma);function_index.append(function)

                #High resolution dataset
                dnhr=dhr[dhr['rep']==n]
                dnhr=dnhr[(dnhr['x1']>=-2.0) & (dnhr['x1']<=2.0)]
                dnhr=clean_index(dnhr)
                #-------------------------------------------------------------
                #train_value=1.0
                #train_border_row=dnhr[(dnhr['x1']<=1.0) & (dnhr['x1']>=0.99)] 
                #print(train_border_row)
                #train_size=train_border_row.index[0]
                #print(train_size)
                #-------------------------------------------------------------

                #Get interpolation/extrapolation border
                #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
                n_points=int(len(dhr.index)/10)
                train_fraction=3/4;train_size=int(n_points*train_fraction)
                print(train_size)
                #~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

                #Read BMS data
                filename='BMS_'+function+'_n_'+str(n)+'_sigma_'+str(sigma)+ '_r_' + str(realization) + '_res_'+ resolutions[resolution] +  '_trace_'+str(steps)+'_prior_'+str(NPAR)+ '.csv'
        
                #trace=pd.read_csv('../../data/MSTraces/1x_resolution/' + filename, sep=';', header=None, names=['t', 'H', 'expr', 'parvals', 'kk1', 'kk2','kk3'])
                trace_path='../../data/MSTraces/' + resolution + '_resolution/'
                trace=pd.read_csv(trace_path + filename, sep=';', header=None, names=['t', 'H', 'expr', 'parvals', 'kk1', 'kk2','kk3'])
                dinterpolate=add_bms_pred(dnhr, trace, NPAR)

                #Remove approximation points
                #-------------------------------------------------------------

                significant_figures=2
                if resolution=='4e-3x' or '2x': #It seems it is better to consider 3 for very high resolutions
                        significant_figures=3
                    
                dinterpolate=dinterpolate.round({'x1':significant_figures})
                dn=d[d['rep']==n]
                dn.set_index('Unnamed: 0', inplace=True)
                dn.index.name = None
                dn=dn.reset_index(drop=True)
                dn=dn.round({'x1':significant_figures})
                dinterpolate=dinterpolate[-dinterpolate['x1'].isin(dn['x1'].tolist() )] #cross dataframes to get only high resolution points
                #-------------------------------------------------------------

                #display(dinterpolate.loc[train_size-1])
            
                print("interpolation")
                display(dinterpolate.loc[train_size -5 : train_size - 1])
                
                print("extrapolation")
                display(dinterpolate.loc[train_size -1 : train_size + 4])
                
                #Errors
                #-----------------------------------------------------------------------------------------------------------------------
                #nns
                rmse_nn_train_i=root_mean_squared_error(dinterpolate.loc[:train_size-1]['ymodel'],dinterpolate.loc[:train_size -1]['y'])
                rmse_nn_train.append(rmse_nn_train_i)
                
                rmse_nn_test_i=root_mean_squared_error(dinterpolate.loc[train_size-1:]['ymodel'],dinterpolate.loc[train_size -1:]['y'])
                rmse_nn_test.append(rmse_nn_test_i)

                mae_nn_train_i=mean_absolute_error(dinterpolate.loc[:train_size-1]['ymodel'],dinterpolate.loc[:train_size -1]['y'])
                mae_nn_train.append(mae_nn_train_i)
            
                mae_nn_test_i=mean_absolute_error(dinterpolate.loc[train_size-1:]['ymodel'],dinterpolate.loc[train_size -1:]['y'])
                mae_nn_test.append(mae_nn_test_i)

                #bms
                try:
                    rmse_mdl_train_i=root_mean_squared_error(dinterpolate.loc[:train_size-1]['ybms'],dinterpolate.loc[:train_size-1]['y'])
                except ValueError:
                    rmse_mdl_train_i=np.inf
                rmse_mdl_train.append(rmse_mdl_train_i)

                try:
                    rmse_mdl_test_i=root_mean_squared_error(dinterpolate.loc[train_size-1:]['ybms'],dinterpolate.loc[train_size-1:]['y'])
                except ValueError:
                    rmse_mdl_test_i=np.inf
                    
                rmse_mdl_test.append(rmse_mdl_test_i)

                try:
                    mae_mdl_train_i=mean_absolute_error(dinterpolate.loc[:train_size-1]['ybms'],dinterpolate.loc[:train_size -1]['y'])
                except ValueError:
                    mae_mdl_train_i=np.inf
                    
                mae_mdl_train.append(mae_mdl_train_i)

                try:
                    mae_mdl_test_i=mean_absolute_error(dinterpolate.loc[train_size-1:]['ybms'],dinterpolate.loc[train_size -1:]['y'])
                except ValueError:
                    mae_mdl_test_i=np.inf
                
                mae_mdl_test.append(mae_mdl_test_i)
                #-----------------------------------------------------------------------------------------------------------------------              
             

              
#Save all in a dataframe      
errors_df=pd.DataFrame({'sigma':sigma_index, 'function':function_index, 'mae_nn_interp.':mae_nn_train, 'mae_nn_extrap.':mae_nn_test, 
                        'mae_mdl_interp.':mae_mdl_train, 'mae_mdl_extrap.':mae_mdl_test, 'rmse_nn_interp.':rmse_nn_train, 
                        'rmse_nn_extrap.': rmse_nn_test, 'rmse_mdl_interp.':rmse_mdl_train, 'rmse_mdl_extrap.': rmse_mdl_test, 
                        'n':n_index, 'r': r_index})
errors_df.to_csv('../../data/errors_interpolation_' + resolution + '.csv')
display(errors_df)

3750


<lambdifygenerated-27>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + x1**x1)))))/x1
<lambdifygenerated-28>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + x1**x1)))))/x1
<lambdifygenerated-29>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + (2*x1)**x1)))))/x1
<lambdifygenerated-30>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + (2*x1)**x1)))))/x1
<lambdifygenerated-31>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + (2*x1)**x1)))))/x1
<lambdifygenerated-32>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + (2*x1)**x1)))))/x1
<lambdifygenerated-33>:2: RuntimeWarning: invalid value encountered in power
  return (x1 - tan(x1*tanh(x1*(x1 + tanh(x1 + (x1 + x1**x1)**x1)))))/x1
<lambdi

interpolation


<lambdifygenerated-81>:2: RuntimeWarning: invalid value encountered in sqrt
  return (_a4_*(sqrt(x1) + x1) - tan(_a3_*tanh(_a1_*(_a2_*_a6_ + tanh(_a5_ + (-_a6_ + x1)*(x1 + abs(_a6_/_a5_)**x1)**(_a5_ - _a7_)/_a7_)))))/x1
<lambdifygenerated-81>:2: RuntimeWarning: invalid value encountered in power
  return (_a4_*(sqrt(x1) + x1) - tan(_a3_*tanh(_a1_*(_a2_*_a6_ + tanh(_a5_ + (-_a6_ + x1)*(x1 + abs(_a6_/_a5_)**x1)**(_a5_ - _a7_)/_a7_)))))/x1
<lambdifygenerated-82>:2: RuntimeWarning: invalid value encountered in sqrt
  return (_a4_*(sqrt(x1) + x1) - tan(_a3_*tanh(_a1_*(_a2_*_a6_ + tanh(_a5_ + (-_a6_ + x1)*(x1 + abs(_a6_/_a5_)**x1)**(_a5_ - _a7_)/_a7_)))))/x1
<lambdifygenerated-83>:2: RuntimeWarning: invalid value encountered in sqrt
  return (_a4_*(sqrt(2)*sqrt(x1) + x1) - tan(_a3_*tanh(_a1_*(_a2_*_a6_ + tanh(_a5_ + (-_a6_ + x1)*(x1 + abs(_a6_/_a5_)**x1)**(_a5_ - _a7_)/_a7_)))))/x1
<lambdifygenerated-83>:2: RuntimeWarning: invalid value encountered in power
  return (_a4_*(sqrt(2)*sqrt(x1) +

,x1,y,rep,ymodel,ybms
3745,0.997,0.987381,0.0,0.982325,0.987785
3746,0.998,0.987426,0.0,0.982335,0.987832
3747,0.998,0.987470,0.0,0.982345,0.987880
3748,0.999,0.987515,0.0,0.982355,0.987927


...

3750
interpolation


,x1,y,rep,ymodel,ybms
3745,0.997,0.64259,9.0,0.716926,0.633432
3746,0.998,0.64247,9.0,0.716921,0.633157
3747,0.998,0.64235,9.0,0.716917,0.632883
3748,0.999,0.64223,9.0,0.716913,0.632609


extrapolation


,x1,y,rep,ymodel,ybms
3750,1.001,0.64199,9.0,0.716904,0.632059
3751,1.002,0.64187,9.0,0.716900,0.631785
3752,1.002,0.64175,9.0,0.716896,0.631510
3753,1.003,0.64163,9.0,0.716892,0.631235


,sigma,function,mae_nn_interp.,mae_nn_extrap.,mae_mdl_interp.,mae_mdl_extrap.,rmse_nn_interp.,rmse_nn_extrap.,rmse_mdl_interp.,rmse_mdl_extrap.,n,r
0,0.0,tanh,0.000853,0.013224,0.000273,0.019884,0.001288,0.013445,0.000354,0.025308,0,0
1,0.0,tanh,0.001841,0.010059,0.001976,0.116273,0.002560,0.011410,0.002339,0.152121,1,0
2,0.0,tanh,0.000912,0.050137,0.001519,0.298283,0.001101,0.069633,0.001971,0.335729,2,0
3,0.0,tanh,0.000196,0.023162,0.000119,0.174840,0.000230,0.030903,0.000175,2.283758,3,0
4,0.0,tanh,0.000404,0.017901,0.000060,0.010950,0.000509,0.020898,0.000069,0.013071,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...
655,0.2,leaky_ReLU,0.022261,0.118110,0.009094,0.151337,0.031651,0.138461,0.010507,0.172584,5,2
656,0.2,leaky_ReLU,0.015896,0.099963,0.013044,0.079376,0.024994,0.107615,0.017945,0.089668,6,2
657,0.2,leaky_ReLU,0.038772,0.126559,0.031943,0.141442,0.048350,0.150152,0.041721,0.142647,7,2
658,0.2,leaky_ReLU,0.039122,0.176571,0.030463,0.078499,0.050074,0.189842,0.036354,0.085304,8,2
